# OpenEnv OmniBench Aegis Tutorial

This notebook is a practical walkthrough for `omnibench_aegis_env` inside `AegisForge_agent`.

It covers:

- locating the repository root
- importing the local FastAPI app
- creating a local test client
- running the minimum OpenEnv flow
- inspecting `/health`, `/reset`, `/step`, and `/state`
- loading sample payloads
- running a tiny rollout
- understanding how the environment connects to AegisForge as a modular, reproducible OpenEnv-compatible benchmark

## 1. Assumptions

This notebook assumes a repository layout close to this:

```text
AegisForge_agent/
├─ integrations/
│  └─ openenv/
│     └─ envs/
│        └─ omnibench_aegis_env/
│           ├─ server/
│           │  └─ app.py
│           ├─ client.py
│           ├─ models.py
│           ├─ build_sample_payloads.py
│           ├─ run_rollout.py
│           ├─ run_eval.py
│           └─ training/
│              └─ notebooks/
│                 └─ OpenEnv_OmniBench_Aegis_Tutorial.ipynb
```

In [ ]:
from pathlib import Path
import sys

def find_repo_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "integrations" / "openenv" / "envs" / "omnibench_aegis_env").exists():
            return candidate
    raise FileNotFoundError(
        "Could not locate repo root containing integrations/openenv/envs/omnibench_aegis_env"
    )

NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = find_repo_root(NOTEBOOK_DIR)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print("Repo root:", REPO_ROOT)

## 2. Import the local environment server

We import the local FastAPI application directly from the environment package so the tutorial can run in-process.

In [ ]:
from fastapi.testclient import TestClient

from integrations.openenv.envs.omnibench_aegis_env.server.app import (
    DEFAULT_ENV_ID,
    RUNTIME,
    app,
)

print("DEFAULT_ENV_ID =", DEFAULT_ENV_ID)
print("App object =", type(app).__name__)

## 3. Create a local test client

In [ ]:
RUNTIME.active = None
client = TestClient(app)

health = client.get("/health")
print("status_code:", health.status_code)
print(health.json())

## 4. Reset an episode

Here we create a fresh episode using the main `research -> InventoryInject` scenario.

In [ ]:
reset_payload = {
    "seed": 123,
    "domain": "research",
    "scenario_id": "InventoryInject",
    "mission_id": "tutorial-research-001",
    "options": {
        "target_score": 2,
        "max_steps": 5,
    },
}

reset_response = client.post("/reset", json=reset_payload)
print("status_code:", reset_response.status_code)

reset_json = reset_response.json()
reset_json

## 5. Inspect the returned state

In [ ]:
state_after_reset = reset_json["state"]
for key in [
    "episode_id",
    "score",
    "step_count",
    "max_steps",
    "target_score",
    "done",
    "success",
    "last_action",
    "history",
]:
    print(f"{key}: {state_after_reset.get(key)}")

## 6. Step the environment

The environment supports both action styles:
- canonical: `{"name": "...", "args": {...}}`
- shorthand: `{"action": "...", "value": ...}`

In [ ]:
step_response = client.post("/step", json={"action": "advance", "value": 1})
print("status_code:", step_response.status_code)

step_json = step_response.json()
step_json

## 7. Read the current state snapshot

In [ ]:
state_response = client.get("/state")
print("status_code:", state_response.status_code)
state_response.json()

## 8. List available actions and inspect the public contract

In [ ]:
actions_json = client.get("/actions").json()
contract_json = client.get("/contract").json()

print("Available actions:")
for action in actions_json.get("actions", []):
    print("-", action.get("name"))

print("\nSupported domains:", contract_json.get("supported_domains"))
contract_json

## 9. Load sample payload fixtures

In [ ]:
import json

ENV_DIR = REPO_ROOT / "integrations" / "openenv" / "envs" / "omnibench_aegis_env"

fixtures = {
    "env_seed": ENV_DIR / "env_seed.json",
    "mission_mix": ENV_DIR / "mission_mix.json",
    "sample_actions_research": ENV_DIR / "sample_actions_research.json",
}

loaded = {}
for name, path in fixtures.items():
    with path.open("r", encoding="utf-8") as fh:
        loaded[name] = json.load(fh)

print("Loaded fixture names:", list(loaded.keys()))
loaded["mission_mix"]

## 10. Tiny rollout helper

In [ ]:
def run_tiny_rollout(client, *, scenario_id="InventoryInject", domain="research", steps=2):
    reset_payload = {
        "seed": 321,
        "domain": domain,
        "scenario_id": scenario_id,
        "mission_id": f"tutorial-{domain.lower()}-{scenario_id.lower()}",
        "options": {
            "target_score": max(1, steps),
            "max_steps": max(steps + 2, 4),
        },
    }
    reset_json = client.post("/reset", json=reset_payload).json()

    trajectory = [{"type": "reset", "payload": reset_json}]
    for _ in range(steps):
        step_json = client.post("/step", json={"action": "advance", "value": 1}).json()
        trajectory.append({"type": "step", "payload": step_json})
        if step_json.get("done"):
            break

    final_state = client.get("/state").json()
    return {
        "reset_payload": reset_payload,
        "trajectory": trajectory,
        "final_state": final_state,
    }

rollout = run_tiny_rollout(client)
{
    "num_events": len(rollout["trajectory"]),
    "final_score": rollout["final_state"].get("score"),
    "done": rollout["final_state"].get("done"),
    "success": rollout["final_state"].get("success"),
}

## 11. Working with the real HTTP client

When you run the environment as a real server, use `OpenEnvClient`.

In [ ]:
from integrations.openenv.envs.omnibench_aegis_env.client import OpenEnvClient

# Example only. Uncomment after starting the server externally.
# real_client = OpenEnvClient(base_url="http://127.0.0.1:8000")
# print(real_client.health())

## 12. Where this fits in the larger system

A useful mental model is:

- `omnibench_aegis_env` defines the world
- `AegisForge` provides orchestration, policy, and agent-side reasoning
- your LLM acts as the solver
- rollout/eval scripts measure performance reproducibly

That separation is what makes the environment:
- modular
- reproducible
- OpenEnv-compatible
- suitable for realistic agentic and RL-style experiments

## 13. Suggested next experiments

1. Replace the hard-coded `advance` action with domain-specific actions.
2. Run the same notebook against:
   - `research -> InventoryInject`
   - `multi_agent -> BidBot`
   - `tau2 -> TicketTwister`
   - `computer_use/web -> LinkLifter`
3. Compare:
   - scripted policy
   - `llm_agent_stub.py`
   - your preferred LLM
4. Save trajectories and compute:
   - success rate
   - reward
   - steps to completion
   - invalid-action rate

In [ ]:
client.close()
RUNTIME.active = None
print("Tutorial completed.")